## Imports and Configuration

In [ ]:
import pandas as pd
from sqlalchemy.types import Date
from sqlalchemy import create_engine, inspect, URL
from snowflake.connector.pandas_tools import pd_writer
from urllib.parse import quote_plus

# --- MYSQL CONFIG ---
MYSQL_DB = {
    "user": "<user_name>",
    "pass": "<password>",
    "host": "<host_name>",
    "port": "3306",
    "db": "<database_name>"
}

# --- SNOWFLAKE CONFIG ---
SNOW_CONF = {
    "account": "<account_identifier>",
    "user": "<user_name>",
    "password": "<password>",
    "database": "<database_name>",
    "schema": "<schema_name>",
    "warehouse": "<compute_name>",
    "role": "<role_name>"
}

print("Libraries imported and config set")

## Create Engines and Test Connections

In [ ]:
# Create MySQL Engine
mysql_pass = quote_plus(MYSQL_DB['pass'])
mysql_url = f"mysql+pymysql://{MYSQL_DB['user']}:{mysql_pass}@{MYSQL_DB['host']}:{MYSQL_DB['port']}/{MYSQL_DB['db']}"
mysql_engine = create_engine(mysql_url)

# Create Snowflake Engine
snow_url = URL(
    account=SNOW_CONF['account'],
    user=SNOW_CONF['user'],
    password=SNOW_CONF['password'],
    database=SNOW_CONF['database'],
    schema=SNOW_CONF['schema'],
    warehouse=SNOW_CONF['warehouse'],
    role=SNOW_CONF['role']
)
snow_engine = create_engine(snow_url)

# Test
try:
    with mysql_engine.connect() as conn:
        print("MySQL Connection Successful")
    with snow_engine.connect() as conn:
        print("Snowflake Connection Successful")
except Exception as e:
    print(f"Connection Error: {e}")

## Inspect Tables

In [ ]:
inspector = inspect(mysql_engine)
mysql_tables = inspector.get_table_names()

print(f"Found {len(mysql_tables)} tables in MySQL Database : {mysql_tables}")

## The Migration Loop

In [ ]:
for table in mysql_tables:
    try:
        print(f"--- Extracting {table} from MySQL ---")
        df = pd.read_sql_table(table, mysql_engine)
        
        if df.empty:
            continue

        df.columns = [str(c).strip().upper() for c in df.columns]

        dtype_mapping = {}

        for col in df.columns:
            # Target only the date columns
            if 'DATE' in col.upper():
                # Convert to datetime objects, then to date objects (removes the 00:00:00)
                df[col] = pd.to_datetime(df[col], errors='coerce').dt.date
                
                # Fill the NaT/NaN with None for a clean SQL NULL
                df[col] = df[col].where(pd.notnull(df[col]), None)
                
                # Explicitly map to the DATE type (not Timestamp)
                dtype_mapping[col] = Date()

        dest_table = f"stg_{table}"

        print(f"--- Loading {table} to Snowflake ---")
        
        # Use pd_writer for significantly faster cloud loading
        df.to_sql(
            dest_table, 
            snow_engine, 
            if_exists='replace', 
            index=False,
            method=pd_writer,
            dtype=dtype_mapping
        )
        print(f"Successfully migrated {table} -> {dest_table}")
        
    except Exception as e:
        print(f"Failed to migrate {table}: {e}")

print("\n--- ALL TASKS COMPLETE ---")